# Lineare Regression: Von der Lernzeit zur Klausurpunktzahl

## Einführung

In diesem Notebook sagen wir die **Punktzahl einer Klausur** anhand der **Anzahl gelernter Stunden** vorher. Anders als bei Entscheidungsbäumen oder k-Means sagen wir hier keine Kategorie voraus, sondern eine kontinuierliche Zahl – das ist der Kern der **linearen Regression**.

Wir legen dazu einen kleinen, selbst erstellten Beispieldatensatz an (kein Download nötig) und trainieren damit ein lineares Regressionsmodell mit scikit-learn.

Zuerst installieren wir die benötigten Bibliotheken:

In [ ]:
%pip install pandas

In [ ]:
%pip install scikit-learn

In [ ]:
%pip install matplotlib

In [ ]:
# Benötigte Bibliotheken importieren
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

### Schritt 1: Datensatz erstellen

Wir erstellen selbst einen kleinen Datensatz: Für 8 (fiktive) Schüler kennen wir die Anzahl der gelernten Stunden und die erreichte Klausurpunktzahl (von maximal 100 Punkten). Damit die Daten etwas realistischer wirken, fügen wir ein leichtes zufälliges Rauschen hinzu.

In [ ]:
# Zufallsgenerator festlegen, damit die Ergebnisse reproduzierbar sind
np.random.seed(42)

# Lernstunden der Schüler
lernstunden = np.array([1, 2, 3, 4, 5, 6, 7, 8])

# Punktzahl mit leichtem Rauschen: pro Stunde ca. 7 Punkte mehr, plus Grundpunktzahl 30
rauschen = np.random.normal(loc=0, scale=4, size=len(lernstunden))
punktzahl = 30 + 7 * lernstunden + rauschen

# In einem DataFrame zusammenfassen
df = pd.DataFrame({
    'Lernstunden': lernstunden,
    'Punktzahl': punktzahl
})

print(df.head())

### Schritt 2: Datenvisualisierung

Bevor wir ein Modell trainieren, schauen wir uns die Daten in einem Streudiagramm an.

In [ ]:
# Streudiagramm: Lernstunden vs. Punktzahl
plt.figure(figsize=(8, 6))
plt.scatter(df['Lernstunden'], df['Punktzahl'], s=100)
plt.title('Lernstunden vs. Klausurpunktzahl')
plt.xlabel('Lernstunden')
plt.ylabel('Punktzahl')
plt.show()

### Aufgabe 1:
Betrachte das Streudiagramm. Was erwartest du für den Zusammenhang zwischen Lernstunden und Punktzahl? Beschreibe ihn in eigenen Worten (z. B.: Steigt die Punktzahl eher gleichmäßig, stark schwankend oder gar nicht mit den Lernstunden?).

## Schritt 3: Lineares Regressionsmodell trainieren

Jetzt lassen wir scikit-learn die am besten passende Gerade (nach der Methode der kleinsten Quadrate) berechnen. Dazu brauchen wir X (die Eingabe, hier die Lernstunden) und y (die Ausgabe, hier die Punktzahl).

In [ ]:
# X muss zweidimensional sein (Zeilen = Datenpunkte, Spalten = Merkmale).
# Die doppelten eckigen Klammern df[['Lernstunden']] sorgen dafür, dass wir
# ein DataFrame (2D-Tabelle) bekommen statt einer Series (1D-Liste).
X = df[['Lernstunden']]
y = df['Punktzahl']

# Modell erstellen und trainieren
modell = LinearRegression()
modell.fit(X, y)

# Steigung (m) und Achsenabschnitt (b) ausgeben
print(f"Steigung (m): {modell.coef_[0]:.2f}")
print(f"Achsenabschnitt (b): {modell.intercept_:.2f}")

## Schritt 4: Regressionsgerade visualisieren

Wir zeichnen die vom Modell gefundene Gerade zusammen mit den echten Datenpunkten in ein Diagramm.

In [ ]:
# Vorhersagen für alle Trainingspunkte berechnen
y_vorhersage = modell.predict(X)

plt.figure(figsize=(8, 6))
plt.scatter(df['Lernstunden'], df['Punktzahl'], s=100, label='Datenpunkte')
plt.plot(df['Lernstunden'], y_vorhersage, color='red', label='Regressionsgerade')
plt.title('Lineare Regression: Lernstunden vs. Klausurpunktzahl')
plt.xlabel('Lernstunden')
plt.ylabel('Punktzahl')
plt.legend()
plt.show()

## Schritt 5: Güte des Modells (R²)

Das Bestimmtheitsmaß R² sagt uns, wie gut die Gerade die Daten erklärt: 0 bedeutet, das Modell erklärt nichts, 1 bedeutet eine perfekte Vorhersage.

In [ ]:
r2 = r2_score(y, y_vorhersage)
print(f"R² des Modells: {r2:.3f}")

### Aufgabe 2:
Berechne mit dem trainierten Modell eine Vorhersage für einen Schüler, der **9 Stunden** gelernt hat.

`modell.predict(...)` erwartet die Eingabe im **gleichen Format** wie beim Training – also ein DataFrame mit einer Spalte `'Lernstunden'`, auch wenn du nur einen einzigen neuen Wert vorhersagen möchtest. So berechnest du zum Beispiel die Vorhersage für einen Schüler, der **5 Stunden** gelernt hat:

```python
neue_daten = pd.DataFrame({'Lernstunden': [5]})
vorhersage = modell.predict(neue_daten)
print(f"Vorhergesagte Punktzahl bei 5 Stunden: {vorhersage[0]:.2f}")
```

Die Code-Zelle unten enthält bereits genau diesen Code. Ändere **nur die markierte Zahl** (`neue_stunden = 5`) auf `9` und führe die Zelle erneut aus.

In [ ]:
# Vorhersage für eine neue Lernstundenzahl berechnen
neue_stunden = 5  # <-- HIER ÄNDERST DU: auf 9 setzen

neue_daten = pd.DataFrame({'Lernstunden': [neue_stunden]})
vorhersage = modell.predict(neue_daten)
print(f"Vorhergesagte Punktzahl bei {neue_stunden} Stunden: {vorhersage[0]:.2f}")

### Aufgabe 3: Überanpassung mit einem Polynom (Overfitting)

Statt einer Geraden kann man auch ein Polynom höheren Grades an die Daten anpassen, z. B. mit Grad 8. Dazu verwendet man `PolynomialFeatures` aus `sklearn.preprocessing`, um zusätzliche Spalten (x², x³, ..., x⁸) zu erzeugen. `poly.fit_transform(X)` berechnet dabei zunächst, welche neuen Spalten benötigt werden (`fit`), und wendet sie sofort auf `X` an (`transform`) – in einem Schritt. Auf den so erweiterten Spalten trainiert man wieder ganz normal ein `LinearRegression`-Modell.

Damit die gefundene Kurve nicht nur an den 8 Trainingspunkten sichtbar wird, sondern als durchgehende Linie, erzeugt der Code unten mit `np.linspace(1, 8, 200)` zusätzlich 200 gleichmäßig verteilte x-Werte zwischen 1 und 8. Für diese neuen x-Werte wird **nur** `poly.transform(...)` verwendet (nicht `fit_transform`), da `poly` bereits beim Training auf den echten Daten eingerichtet wurde – wir wollen ja dieselben Spalten (x², x³, ...) berechnen, nicht neu festlegen, welche Spalten es überhaupt gibt.

Der Code unten ist bereits vollständig lauffähig und mit `grad = 8` vorbereitet – führe ihn zunächst so aus, wie er ist. Beobachte im Diagramm, wie sich die grüne Polynom-Kurve eng an die Trainingspunkte anschmiegt und sich zwischen ihnen „schlängelt“.

Ändere anschließend **nur die markierte Zeile** (`grad = 8`) und probiere der Reihe nach `grad = 1`, `grad = 2` und `grad = 4` aus. Beobachte dabei:
1. Bei welchem Grad ähnelt die grüne Kurve am ehesten der roten Regressionsgeraden?
2. Bei welchem Grad „schlängelt“ sich die Kurve am stärksten zwischen den Punkten?
3. Warum wäre ausgerechnet die am stärksten schlängelnde Kurve bei der Vorhersage für neue, unbekannte Lernstunden (z. B. 9 oder 10 Stunden) vermutlich am schlechtesten? Welches Problem aus Kapitel „KI-Systeme: Training und Optimierung“ (Stichwort Über-/Unteranpassung) erkennst du hier wieder?

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

grad = 8  # <-- HIER ÄNDERST DU: probiere z. B. auch 1, 2 oder 4

# Polynom-Merkmale erzeugen (x, x^2, x^3, ..., x^grad) und Modell trainieren
poly = PolynomialFeatures(degree=grad)
X_poly = poly.fit_transform(X)

modell_poly = LinearRegression()
modell_poly.fit(X_poly, y)

# 200 gleichmäßig verteilte x-Werte zwischen 1 und 8 für eine glatte Kurve erzeugen
x_kurve_werte = np.linspace(1, 8, 200)
x_kurve = pd.DataFrame({'Lernstunden': x_kurve_werte})
y_kurve = modell_poly.predict(poly.transform(x_kurve))

# Vergleich: Datenpunkte, lineare Regressionsgerade und Polynom-Kurve
plt.figure(figsize=(8, 6))
plt.scatter(df['Lernstunden'], df['Punktzahl'], s=100, label='Datenpunkte')
plt.plot(df['Lernstunden'], y_vorhersage, color='red', label='Lineare Regression (Grad 1)')
plt.plot(x_kurve_werte, y_kurve, color='green', label=f'Polynom (Grad {grad})')
plt.title('Vergleich: Lineare Regression vs. Polynom')
plt.xlabel('Lernstunden')
plt.ylabel('Punktzahl')
plt.legend()
plt.show()

## Zusammenfassung

In diesem Notebook haben wir folgende Schritte durchgeführt:
1. Einen kleinen, synthetischen Datensatz (Lernstunden und Klausurpunktzahl) erstellt
2. Die Daten in einem Streudiagramm visualisiert
3. Ein lineares Regressionsmodell trainiert und Steigung sowie Achsenabschnitt bestimmt
4. Die Regressionsgerade zusammen mit den Datenpunkten geplottet
5. Die Modellgüte mit R² bewertet
6. Eine Vorhersage für einen neuen Wert berechnet
7. Mit einem Polynom hohen Grades experimentiert und Overfitting beobachtet